# 数据集整合

整合 `colas.csv` 和 `hall.csv` 两个数据集，提取所需列并重新编号受试者ID

In [21]:
# 导入必要的库
import pandas as pd
import numpy as np
import os
from datetime import datetime

print("Libraries imported successfully")

Libraries imported successfully


In [22]:
# 读取两个数据集
colas_path = '../../SourceData/colas.csv'
hall_path = '../../SourceData/hall.csv'

print("读取 colas.csv...")
df_colas = pd.read_csv(colas_path)
print(f"  - Shape: {df_colas.shape}")
print(f"  - Columns: {df_colas.columns.tolist()}")

print("\n读取 hall.csv...")
df_hall = pd.read_csv(hall_path)
print(f"  - Shape: {df_hall.shape}")
print(f"  - Columns: {df_hall.columns.tolist()}")

print("\n数据读取完成！")

读取 colas.csv...
  - Shape: (114253, 11)
  - Columns: ['Unnamed: 0', 'id', 'time', 'gl', 'gender', 'age', 'BMI', 'glycaemia', 'HbA1c', 'follow.up', 'T2DM']

读取 hall.csv...
  - Shape: (105426, 51)
  - Columns: ['id', 'time', 'gl', 'Age', 'BMI', 'A1C', 'FBG', 'ogtt.2hr', 'insulin', 'hs.CRP', 'Tchol', 'Trg', 'HDL', 'LDL', 'mean_glucose', 'sd_glucose', 'range_glucose', 'min_glucose', 'max_glucose', 'quartile.25_glucose', 'median_glucose', 'quartile.75_glucose', 'mean_slope', 'max_slope', 'number_Random140', 'number_Random200', 'percent_below.80', 'percent_above.130', 'se_glucose_mean', 'numGE', 'mage', 'j_index', 'IQR', 'modd', 'distance_traveled', 'coef_variation', 'number_Random140_normByDays', 'number_Random200_normByDays', 'numGE_normByDays', 'distance_traveled_normByDays', 'diagnosis', 'freq_low', 'freq_moderate', 'freq_severe', 'glucotype', 'Height', 'Weight', 'Insulin_rate_dd', 'perc_cgm_prediabetic_range', 'perc_cgm_diabetic_range', 'SSPG']

数据读取完成！


In [23]:
# 提取所需列并统一列名
# colas.csv: 需要提取 id, time, gl, age, BMI
# hall.csv: 需要提取 id, time, gl, Age, BMI (注意Age是大写A)

print("="*70)
print("提取所需列")
print("="*70)

# 从 colas.csv 提取列（统一列名为小写）
colas_selected = df_colas[['id', 'time', 'gl', 'age', 'BMI']].copy()
colas_selected.columns = ['id', 'time', 'gl', 'age', 'bmi']  # 统一为小写
colas_selected['source'] = 'colas'  # 添加数据源标记

print(f"\ncolas.csv 提取结果:")
print(f"  - Shape: {colas_selected.shape}")
print(f"  - Columns: {colas_selected.columns.tolist()}")
print(f"  - 受试者数量: {colas_selected['id'].nunique()}")

# 从 hall.csv 提取列（统一列名为小写）
hall_selected = df_hall[['id', 'time', 'gl', 'Age', 'BMI']].copy()
hall_selected.columns = ['id', 'time', 'gl', 'age', 'bmi']  # 统一为小写
hall_selected['source'] = 'hall'  # 添加数据源标记

print(f"\nhall.csv 提取结果:")
print(f"  - Shape: {hall_selected.shape}")
print(f"  - Columns: {hall_selected.columns.tolist()}")
print(f"  - 受试者数量: {hall_selected['id'].nunique()}")

print("\n列提取完成！")

提取所需列

colas.csv 提取结果:
  - Shape: (114253, 6)
  - Columns: ['id', 'time', 'gl', 'age', 'bmi', 'source']
  - 受试者数量: 208

hall.csv 提取结果:
  - Shape: (105426, 6)
  - Columns: ['id', 'time', 'gl', 'age', 'bmi', 'source']
  - 受试者数量: 57

列提取完成！


In [24]:
# 重新编号受试者ID，避免重复
print("="*70)
print("重新编号受试者ID")
print("="*70)

# 获取原始ID信息
colas_unique_ids = colas_selected['id'].unique()
hall_unique_ids = hall_selected['id'].unique()

print(f"\n原始ID统计:")
print(f"  - colas.csv 受试者数: {len(colas_unique_ids)}")
print(f"  - hall.csv 受试者数: {len(hall_unique_ids)}")
print(f"  - colas.csv ID范围: {colas_unique_ids.min()} - {colas_unique_ids.max()}")
print(f"  - hall.csv ID示例: {hall_unique_ids[:3]}")

# 为colas数据集创建ID映射（从1开始）
colas_id_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted(colas_unique_ids), start=1)}
colas_selected['new_id'] = colas_selected['id'].map(colas_id_mapping)

# 为hall数据集创建ID映射（从colas的最大ID+1开始）
hall_start_id = len(colas_unique_ids) + 1
hall_id_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted(hall_unique_ids), start=hall_start_id)}
hall_selected['new_id'] = hall_selected['id'].map(hall_id_mapping)

print(f"\n新ID分配:")
print(f"  - colas.csv: 1 - {len(colas_unique_ids)}")
print(f"  - hall.csv: {hall_start_id} - {hall_start_id + len(hall_unique_ids) - 1}")
print(f"  - 总受试者数: {len(colas_unique_ids) + len(hall_unique_ids)}")

# 替换旧ID为新ID
colas_selected['old_id'] = colas_selected['id']
colas_selected['id'] = colas_selected['new_id']
colas_selected.drop('new_id', axis=1, inplace=True)

hall_selected['old_id'] = hall_selected['id']
hall_selected['id'] = hall_selected['new_id']
hall_selected.drop('new_id', axis=1, inplace=True)

print("\nID重新编号完成！")

重新编号受试者ID

原始ID统计:
  - colas.csv 受试者数: 208
  - hall.csv 受试者数: 57
  - colas.csv ID范围: 1 - 208
  - hall.csv ID示例: ['1636-69-001' '1636-69-026' '1636-69-028']

新ID分配:
  - colas.csv: 1 - 208
  - hall.csv: 209 - 265
  - 总受试者数: 265

ID重新编号完成！


In [25]:
# 合并两个数据集
print("="*70)
print("合并数据集")
print("="*70)

# 合并数据
df_merged = pd.concat([colas_selected, hall_selected], ignore_index=True)

print(f"\n合并后数据集信息:")
print(f"  - Total rows: {len(df_merged):,}")
print(f"  - Total subjects: {df_merged['id'].nunique()}")
print(f"  - Columns: {df_merged.columns.tolist()}")
print(f"\n数据来源统计:")
print(df_merged['source'].value_counts())

# 显示数据预览
print(f"\n合并数据集前5行:")
print(df_merged.head())

print(f"\n数据类型:")
print(df_merged.dtypes)

合并数据集

合并后数据集信息:
  - Total rows: 219,679
  - Total subjects: 265
  - Columns: ['id', 'time', 'gl', 'age', 'bmi', 'source', 'old_id']

数据来源统计:
source
colas    114253
hall     105426
Name: count, dtype: int64

合并数据集前5行:
   id                 time    gl   age   bmi source old_id
0   1  2012-01-01 00:00:00  86.0  77.0  25.4  colas      1
1   1  2012-01-01 00:05:00  81.0  77.0  25.4  colas      1
2   1  2012-01-01 00:10:00  78.0  77.0  25.4  colas      1
3   1  2012-01-01 00:15:00  76.0  77.0  25.4  colas      1
4   1  2012-01-01 00:20:00  76.0  77.0  25.4  colas      1

数据类型:
id          int64
time       object
gl        float64
age       float64
bmi       float64
source     object
old_id     object
dtype: object


In [26]:
# 数据质量检查
print("="*70)
print("数据质量检查")
print("="*70)

# 检查缺失值
print("\n缺失值统计:")
missing_stats = df_merged.isnull().sum()
print(missing_stats)
print(f"\n缺失值百分比:")
print((missing_stats / len(df_merged) * 100).round(2))

# 检查ID是否有重复
print(f"\nID唯一性检查:")
print(f"  - 总行数: {len(df_merged)}")
print(f"  - 唯一ID数: {df_merged['id'].nunique()}")
print(f"  - ID是否唯一: {'否，存在重复' if df_merged.duplicated(subset=['id', 'time']).any() else '是（在同一时间点）'}")

# 数据统计摘要
print(f"\n数值列统计摘要:")
print(df_merged[['id', 'gl', 'age', 'bmi']].describe())

数据质量检查

缺失值统计:
id        0
time      0
gl        9
age       0
bmi       0
source    0
old_id    0
dtype: int64

缺失值百分比:
id        0.0
time      0.0
gl        0.0
age       0.0
bmi       0.0
source    0.0
old_id    0.0
dtype: float64

ID唯一性检查:
  - 总行数: 219679
  - 唯一ID数: 265
  - ID是否唯一: 是（在同一时间点）

数值列统计摘要:
                  id             gl            age            bmi
count  219679.000000  219670.000000  219679.000000  219679.000000
mean      167.556949     102.160650      54.506653      28.387948
std        80.392779      22.499224      13.038633       4.892124
min         1.000000      40.000000      25.000000      18.100000
25%       100.000000      87.000000      48.000000      25.400000
50%       200.000000      99.000000      57.000000      27.800000
75%       236.000000     113.000000      64.000000      30.700000
max       265.000000     318.000000      88.000000      48.700000


In [27]:
# 过滤记录时间不足1天的受试者
print("="*70)
print("过滤短时间记录")
print("="*70)

# 确保时间列为datetime格式以便计算
if not pd.api.types.is_datetime64_any_dtype(df_merged['time']):
    df_merged['time'] = pd.to_datetime(df_merged['time'])

# 计算每个受试者的记录时长
# 按照ID分组，计算最大时间减去最小时间
duration_per_id = df_merged.groupby('id')['time'].agg(lambda x: x.max() - x.min())

# 找出记录时间小于1天的受试者ID
# pd.Timedelta(days=1) 表示1天的时间间隔
short_record_ids = duration_per_id[duration_per_id < pd.Timedelta(days=1)].index.tolist()

print(f"\n记录时长统计:")
print(f"  - 最短记录时长: {duration_per_id.min()}")
print(f"  - 最长记录时长: {duration_per_id.max()}")
print(f"  - 记录不足1天的受试者数量: {len(short_record_ids)}")

if len(short_record_ids) > 0:
    print(f"  - 将移除的受试者ID: {short_record_ids}")
    
    # 记录移除前的数据量
    original_rows = len(df_merged)
    original_subjects = df_merged['id'].nunique()
    
    # 移除这些受试者的数据
    df_merged = df_merged[~df_merged['id'].isin(short_record_ids)]
    
    # 记录移除后的数据量
    current_rows = len(df_merged)
    current_subjects = df_merged['id'].nunique()
    
    print(f"\n过滤执行结果:")
    print(f"  - 移除行数: {original_rows - current_rows}")
    print(f"  - 移除受试者数: {original_subjects - current_subjects}")
    print(f"  - 剩余受试者数: {current_subjects}")
else:
    print("\n所有受试者的记录时间均超过1天，无需过滤。")

过滤短时间记录

记录时长统计:
  - 最短记录时长: 0 days 23:52:21
  - 最长记录时长: 423 days 11:25:54
  - 记录不足1天的受试者数量: 17
  - 将移除的受试者ID: [41, 72, 75, 82, 87, 89, 97, 111, 121, 123, 148, 150, 159, 186, 197, 201, 202]

过滤执行结果:
  - 移除行数: 4700
  - 移除受试者数: 17
  - 剩余受试者数: 248


In [28]:
# 数据重采样与严格过滤
print("="*70)
print("数据重采样与严格过滤")
print("="*70)

# 1. 识别并移除存在大间隔(>10分钟)的受试者
print("正在检查数据间隔并进行重采样...")
subjects_with_gaps = []
valid_subjects_data = []

# 按受试者处理
for subject_id, group in df_merged.groupby('id'):
    # 确保按时间排序并去重
    group = group.sort_values('time').drop_duplicates(subset='time')
    
    # 计算时间差
    time_diffs = group['time'].diff().dropna()
    
    # 检查是否存在超过10分钟的间隔
    if (time_diffs > pd.Timedelta(minutes=10)).any():
        subjects_with_gaps.append(subject_id)
        continue

    # 如果没有大间隔，进行重采样
    # 设置索引为时间
    group = group.set_index('time')
    
    # 创建目标5分钟时间网格 (对齐到整点)
    # floor('5min') 确保对齐到 00, 05, 10...
    start_time = group.index.min().floor('5min')
    end_time = group.index.max().floor('5min')
    target_index = pd.date_range(start=start_time, end=end_time, freq='5min')
    target_index.name = 'time'
    
    # 为了准确插值，将目标索引与原始索引合并
    combined_index = group.index.union(target_index).sort_values()
    
    # 重建索引并进行时间插值
    # method='time' 根据时间距离进行线性插值，比默认的线性插值更准确
    group_interp = group.reindex(combined_index)
    group_interp['gl'] = group_interp['gl'].interpolate(method='time')
    
    # 只保留目标5分钟网格的数据
    group_resampled = group_interp.loc[target_index].copy()
    
    # 恢复/填充非时序列 (age和bmi是常量)
    group_resampled['id'] = subject_id
    group_resampled['age'] = group['age'].iloc[0]
    group_resampled['bmi'] = group['bmi'].iloc[0]
    
    # 重置索引
    group_resampled = group_resampled.reset_index()
    
    valid_subjects_data.append(group_resampled)

# 打印过滤结果
print(f"\n过滤结果:")
print(f"  - 原始受试者数: {df_merged['id'].nunique()}")
print(f"  - 因间隔>10分钟被移除的受试者数: {len(subjects_with_gaps)}")
if len(subjects_with_gaps) > 0:
    print(f"  - 移除的ID示例: {subjects_with_gaps[:10]}")

# 更新df_merged
if valid_subjects_data:
    df_merged = pd.concat(valid_subjects_data, ignore_index=True)
    # 确保列顺序
    df_merged = df_merged[['id', 'time', 'gl', 'age', 'bmi']]
    
    print(f"\n重采样后数据集信息:")
    print(f"  - 剩余受试者数: {df_merged['id'].nunique()}")
    print(f"  - 总行数: {len(df_merged):,}")
    
    # 再次检查采样频率
    print("\n频率一致性检查:")
    # 检查所有相邻点的时间差是否都为5分钟
    # 注意：groupby().diff() 会在每个组的第一个元素产生NaT，需要排除
    time_diffs = df_merged.sort_values(['id', 'time']).groupby('id')['time'].diff().dropna()
    is_5min = (time_diffs == pd.Timedelta(minutes=5)).all()
    
    print(f"  - 所有间隔均为5分钟: {is_5min}")
    if not is_5min:
        print("  - 警告: 仍存在非5分钟间隔，请检查代码逻辑。")
        # 打印异常值
        abnormal_diffs = time_diffs[time_diffs != pd.Timedelta(minutes=5)]
        print(f"  - 异常间隔示例:\n{abnormal_diffs.head()}")
else:
    print("\n警告: 所有受试者都被移除了！")
    df_merged = pd.DataFrame(columns=['id', 'time', 'gl', 'age', 'bmi'])

数据重采样与严格过滤
正在检查数据间隔并进行重采样...

过滤结果:
  - 原始受试者数: 248
  - 因间隔>10分钟被移除的受试者数: 80
  - 移除的ID示例: [1, 4, 6, 9, 19, 20, 24, 30, 43, 59]

重采样后数据集信息:
  - 剩余受试者数: 168
  - 总行数: 101,600

频率一致性检查:
  - 所有间隔均为5分钟: True


In [29]:
# 准备导出的最终数据集（仅保留所需列）
print("="*70)
print("准备导出数据")
print("="*70)

# 选择最终要导出的列：id, time, gl, age, bmi
df_export = df_merged[['id', 'time', 'gl', 'age', 'bmi']].copy()

# 确保时间列格式正确
df_export['time'] = pd.to_datetime(df_export['time'])

# 按照id和time排序
df_export = df_export.sort_values(['id', 'time']).reset_index(drop=True)

print(f"\n最终导出数据集信息:")
print(f"  - Shape: {df_export.shape}")
print(f"  - Columns: {df_export.columns.tolist()}")
print(f"  - Total subjects: {df_export['id'].nunique()}")
print(f"  - Date range: {df_export['time'].min()} to {df_export['time'].max()}")

print(f"\n前10行数据:")
print(df_export.head(10))

print(f"\n后10行数据:")
print(df_export.tail(10))

准备导出数据

最终导出数据集信息:
  - Shape: (101600, 5)
  - Columns: ['id', 'time', 'gl', 'age', 'bmi']
  - Total subjects: 168
  - Date range: 2012-01-01 00:00:00 to 2017-05-25 18:05:00

前10行数据:
   id                time     gl   age   bmi
0   2 2012-01-01 00:00:00  167.0  42.0  30.0
1   2 2012-01-01 00:05:00  163.0  42.0  30.0
2   2 2012-01-01 00:10:00  158.0  42.0  30.0
3   2 2012-01-01 00:15:00  151.0  42.0  30.0
4   2 2012-01-01 00:20:00  144.0  42.0  30.0
5   2 2012-01-01 00:25:00  137.0  42.0  30.0
6   2 2012-01-01 00:30:00  132.0  42.0  30.0
7   2 2012-01-01 00:35:00  127.0  42.0  30.0
8   2 2012-01-01 00:40:00  125.0  42.0  30.0
9   2 2012-01-01 00:45:00  125.0  42.0  30.0

后10行数据:
         id                time          gl   age   bmi
101590  258 2017-05-25 17:20:00  120.853333  35.0  26.3
101591  258 2017-05-25 17:25:00  121.853333  35.0  26.3
101592  258 2017-05-25 17:30:00  122.000000  35.0  26.3
101593  258 2017-05-25 17:35:00  121.146667  35.0  26.3
101594  258 2017-05-25 17:40:00  1

In [30]:
# 导出合并后的数据集到CSV
print("="*70)
print("导出数据到CSV")
print("="*70)

# 确保输出目录存在
output_dir = '../DataFormat'
os.makedirs(output_dir, exist_ok=True)

# 生成输出文件名（包含时间戳）
timestamp = datetime.now().strftime('%Y%m%d')
output_file = os.path.join(output_dir, f'merged_cgm_data_{timestamp}.csv')

# 导出到CSV
# df_export.to_csv(output_file, index=False, encoding='utf-8')

print(f"\n✓ 数据导出成功！")
print(f"  - 文件路径: {output_file}")
#print(f"  - 文件大小: {os.path.getsize(output_file) / 1024 / 1024:.2f} MB")
print(f"  - 总行数: {len(df_export):,}")
print(f"  - 受试者数: {df_export['id'].nunique()}")

# 同时导出一个不带时间戳的版本（方便后续使用）
output_file_simple = os.path.join(output_dir, 'merged_cgm_data.csv')
df_export.to_csv(output_file_simple, index=False, encoding='utf-8')
print(f"\n✓ 同时导出了简化文件名版本:")
print(f"  - 文件路径: {output_file_simple}")

print("\n" + "="*70)
print("数据整合完成！")
print("="*70)

导出数据到CSV

✓ 数据导出成功！
  - 文件路径: ../DataFormat\merged_cgm_data_20251221.csv
  - 总行数: 101,600
  - 受试者数: 168

✓ 同时导出了简化文件名版本:
  - 文件路径: ../DataFormat\merged_cgm_data.csv

数据整合完成！


## 数据整合总结

已成功整合 `colas.csv` 和 `hall.csv` 两个数据集：

### 处理步骤：
1. **列名统一**: 将 `Age` → `age`, `BMI` → `bmi` 统一为小写
2. **列提取**: 保留 `id, time, gl, age, bmi` 五列
3. **ID重编号**: 
   - colas受试者ID: 从1开始
   - hall受试者ID: 从colas最大ID+1开始
   - 确保所有受试者ID唯一，避免冲突
4. **数据合并**: 纵向拼接两个数据集
5. **质量检查**: 检查缺失值、重复值等
6. **数据过滤**: 移除记录时间不足1天的受试者数据
7. **导出CSV**: 保存到 `src/DataFormat/` 目录

### 输出文件：
- `merged_cgm_data.csv`: 合并后的最终数据集
- `merged_cgm_data_YYYYMMDD.csv`: 带时间戳的备份版本